<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2026 Winter Term</strong></center></p>

# Direct implementation of a Neural Network

A complete implementation of a toy Neural Network in 2 dimensions. We will first implement a simple linear classifier.

We then extend the code to a 2-layer Neural Network.

original source (DeepLearning.AI)

https://cs231n.github.io/neural-networks-case-study/


**Generating some data**

Lets generate a classification dataset that is not easily linearly separable. Our favorite example is the spiral dataset, which can be generated as follows:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 100 # number of points per class
D = 2 # dimensionality
K = 3 # number of classes
X = np.zeros((N*K,D)) # data matrix (each row = single example)
y = np.zeros(N*K, dtype='uint8') # class labels
for j in range(K):
  ix = range(N*j,N*(j+1))
  r = np.linspace(0.0,1,N) # radius
  t = np.linspace(j*4,(j+1)*4,N) + np.random.randn(N)*0.2 # theta
  X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
  y[ix] = j
# lets visualize the data:
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
plt.show()

In [ ]:
print('input', X.shape)
print('output', y.shape)

The toy spiral data consists of three classes (blue, red, yellow) that are not linearly separable. Total points: 300

Note that, an svm.SVC(kernel='rbf') kernel may be able to classify a spiral dataset, but often be limited to two classes. Here are some examples:

https://www.kaggle.com/code/martininf1n1ty/svm-classification-spiral

https://pub.aimind.so/using-radial-basis-functions-for-svms-with-python-and-scikit-learn-c935aa06a56e

In this case, we have created a dataset of three classes.

In [ ]:
# initialize parameters randomly; D=2 dimension. K=3 classes
W = 0.01 * np.random.randn(D,K)
b = np.zeros((1,K))
print(W)
print(b)

**Compute the class scores**

Since this is a linear classifier, we can compute all class scores very simply in parallel with a single matrix multiplication:

In [ ]:
# compute class scores for a linear classifier
scores = np.dot(X, W) + b
print(scores.shape)


In this example we have 300 2-D points, so after this multiplication the array scores will have size [300 x 3], where each row gives the class scores corresponding to the 3 classes (blue, red, yellow).

In [ ]:
num_examples = X.shape[0]
# get unnormalized probabilities
exp_scores = np.exp(scores)
# normalize them for each example
probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


We now have an array probs of size [300 x 3], where each row now contains the class probabilities. In particular, since we’ve normalized them every row now sums to one. We can now query for the log probabilities assigned to the correct classes in each example:

In [ ]:
correct_logprobs = -np.log(probs[range(num_examples),y])


In [ ]:
num_examples = X.shape[0]
reg = 1e-3
# compute the loss: average cross-entropy loss and regularization
data_loss = np.sum(correct_logprobs)/num_examples
reg_loss = 0.5*reg*np.sum(W*W)
loss = data_loss + reg_loss

In [ ]:
# compute the gradient on scores
dscores = probs
dscores[range(num_examples),y] -= 1
dscores /= num_examples

# backpropagate the gradient to the parameters (W,b)
dW = np.dot(X.T, dscores)
db = np.sum(dscores, axis=0, keepdims=True)

dW += reg*W # add regularization gradient contribution

In [ ]:
# perform a parameter update
step_size = 1e-0
W += -step_size * dW
b += -step_size * db


**Putting it all together: Training a Softmax Classifier**

Putting all of this together, here is the full code for training a Softmax classifier with Gradient descent:

In [ ]:
#Train a Linear Classifier

# initialize parameters randomly
W = 0.01 * np.random.randn(D,K)
b = np.zeros((1,K))

# some hyperparameters
step_size = 1e-0
reg = 1e-3 # regularization strength

# gradient descent loop
num_examples = X.shape[0]
for i in range(200):

  # evaluate class scores, [N x K]
  scores = np.dot(X, W) + b

  # compute the class probabilities
  exp_scores = np.exp(scores)
  probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True) # [N x K]

  # compute the loss: average cross-entropy loss and regularization
  correct_logprobs = -np.log(probs[range(num_examples),y])
  data_loss = np.sum(correct_logprobs)/num_examples
  reg_loss = 0.5*reg*np.sum(W*W)
  loss = data_loss + reg_loss
  if i % 10 == 0:
    print ("iteration %d: loss %f" % (i, loss))

  # compute the gradient on scores
  dscores = probs
  dscores[range(num_examples),y] -= 1
  dscores /= num_examples

  # backpropagate the gradient to the parameters (W,b)
  dW = np.dot(X.T, dscores)
  db = np.sum(dscores, axis=0, keepdims=True)

  dW += reg*W # regularization gradient

  # perform a parameter update
  W += -step_size * dW
  b += -step_size * db


In [ ]:
# evaluate training set accuracy
scores = np.dot(X, W) + b
predicted_class = np.argmax(scores, axis=1)
print ('training accuracy: %.2f' % (np.mean(predicted_class == y)))

In [ ]:
# Visual plot the resulting classifier
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = np.dot(np.c_[xx.ravel(), yy.ravel()], W) + b
Z = np.argmax(Z, axis=1)
Z = Z.reshape(xx.shape)
fig = plt.figure()
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
#fig.savefig('spiral_linear.png')

**Implement a Neural Network using NumPy only**

Clearly, a linear classifier is inadequate for this dataset and we would like to use a Neural Network. One additional hidden layer will suffice for this toy data. We will now need two sets of weights and biases (for the first and second layers):

In [ ]:
# initialize parameters randomly
h = 4 # no. of neurons in the 1st hidden layer
W = 0.01 * np.random.randn(D,h) # D=2 dimension. K=3 classes
b = np.zeros((1,h))
W2 = 0.01 * np.random.randn(h,K)
b2 = np.zeros((1,K))


In [ ]:
print(W)
print(b)
print(W2)
print(b2)

In [ ]:
# evaluate class scores with a 2-layer Neural Network
# output classes are based on scores (y_pred)
hidden_layer = np.maximum(0, np.dot(X, W) + b) # note, ReLU activation
scores = np.dot(hidden_layer, W2) + b2
print(scores.shape)

Notice that the only change from before is one extra line of code, where we first compute the hidden layer representation and then the scores based on this hidden layer. Crucially, we’ve also added a non-linearity, which in this case is simple ReLU that thresholds the activations on the hidden layer at zero.

In [ ]:
# backpropagate the gradient to the parameters
# first backprop into parameters W2 and b2
dW2 = np.dot(hidden_layer.T, dscores)
db2 = np.sum(dscores, axis=0, keepdims=True)

dhidden = np.dot(dscores, W2.T)

# backprop the ReLU non-linearity
dhidden[hidden_layer <= 0] = 0

# finally into W,b
dW = np.dot(X.T, dhidden)
db = np.sum(dhidden, axis=0, keepdims=True)

In [ ]:
# Putting it alltogether
# initialize parameters randomly - 2 hidden layers
h = 10 # size of neurons; start with 4, then gradually increase
W = 0.01 * np.random.randn(D,h)
b = np.zeros((1,h))
W2 = 0.01 * np.random.randn(h,K)
b2 = np.zeros((1,K))

# some hyperparameters
step_size = 1e-0 # 0.1, 0.01, ....
reg = 1e-3 # regularization strength

# gradient descent loop
num_examples = X.shape[0]
for i in range(10000):

  # evaluate class scores, [N x K]
  hidden_layer = np.maximum(0, np.dot(X, W) + b) # note, ReLU activation
  scores = np.dot(hidden_layer, W2) + b2

  # compute the class probabilities
  exp_scores = np.exp(scores)
  probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True) # [N x K]

  # compute the loss: average cross-entropy loss and regularization
  corect_logprobs = -np.log(probs[range(num_examples),y])
  data_loss = np.sum(corect_logprobs)/num_examples
  reg_loss = 0.5*reg*np.sum(W*W) + 0.5*reg*np.sum(W2*W2)
  loss = data_loss + reg_loss
  if i % 1000 == 0:
    print ("iteration %d: loss %f" % (i, loss))

  # compute the gradient on scores
  dscores = probs
  dscores[range(num_examples),y] -= 1
  dscores /= num_examples

  # backpropagate the gradient to the parameters
  # first backprop into parameters W2 and b2
  dW2 = np.dot(hidden_layer.T, dscores)
  db2 = np.sum(dscores, axis=0, keepdims=True)
  # next backprop into hidden layer
  dhidden = np.dot(dscores, W2.T)
  # backprop the ReLU non-linearity
  dhidden[hidden_layer <= 0] = 0
  # finally into W,b
  dW = np.dot(X.T, dhidden)
  db = np.sum(dhidden, axis=0, keepdims=True)

  # add regularization gradient contribution
  dW2 += reg * W2
  dW += reg * W

  # perform a parameter update
  W += -step_size * dW
  b += -step_size * db
  W2 += -step_size * dW2
  b2 += -step_size * db2

In [ ]:
# evaluate training set accuracy
hidden_layer = np.maximum(0, np.dot(X, W) + b)
scores = np.dot(hidden_layer, W2) + b2
predicted_class = np.argmax(scores, axis=1)
print ('training accuracy: %.2f' % (np.mean(predicted_class == y)))


In [ ]:
# plot the resulting classifier
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = np.dot(np.maximum(0, np.dot(np.c_[xx.ravel(), yy.ravel()], W) + b), W2) + b2
Z = np.argmax(Z, axis=1)
Z = Z.reshape(xx.shape)
fig = plt.figure()
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
#fig.savefig('spiral_net.png')

# REDO the entire problem with NN from torch

The following code may be used a template for other datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 100 # number of points per class
D = 2 # dimensionality
K = 3 # number of classes
X = np.zeros((N*K,D)) # data matrix (each row = single example)
y = np.zeros(N*K, dtype='uint8') # class labels
for j in range(K):
  ix = range(N*j,N*(j+1))
  r = np.linspace(0.0,1,N) # radius
  t = np.linspace(j*4,(j+1)*4,N) + np.random.randn(N)*0.2 # theta
  X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
  y[ix] = j
# lets visualize the data:
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
plt.show()

In [ ]:
print('input', X.shape)
print('output', y.shape)

**Why do we need to convert NumPy arrays to PyTorch tensors?**

Primarily to leverage features essential for deep learning, such as GPU acceleration and automatic differentiation.

The main reasons for this conversion are:

  GPU Acceleration: Tensors can be moved to the GPU to perform computations on specialized hardware, which significantly speeds up numerical processing, particularly in deep learning tasks. NumPy arrays are limited to CPU processing.

  Automatic Differentiation (Autograd): PyTorch tensors can track their computational graph, allowing for automatic calculation of gradients (backpropagation). This feature, provided by PyTorch's Autograd system, is fundamental for training neural networks. NumPy arrays do not offer this built-in capability.

  Integration with the PyTorch Ecosystem: Tensors are the native data structure within the PyTorch framework. While you can use NumPy arrays as inputs, using tensors throughout your workflow ensures seamless integration with PyTorch models, data loaders, and libraries, avoiding unnecessary overhead.

  Optimized Operations: PyTorch implements its own optimized tensor engine for deep learning computations. Converting to tensors allows you to benefit from these specific optimizations.

  Memory Efficiency: When converted on the CPU using functions like torch.from_numpy(), the resulting tensor and the original NumPy array can share the same underlying memory, eliminating the need to copy data and improving efficiency. This memory sharing does not extend to the GPU, however, where a separate memory allocation is required.

In summary, you should use NumPy for general-purpose scientific computing and data analysis on the CPU, and convert to PyTorch tensors when you require deep learning functionalities, GPU power, or gradient computations.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

# Device configuration (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Convert numpy arrays to torch tensors
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long() # Use long for classification labels

# Create a TensorDataset
train_data = TensorDataset(X_tensor, y_tensor)

# Create data loaders for batching and shuffling
batch_size = 16
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
#test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
# don't necessarily have to shuffle the testing data

In [ ]:
print('original input', X.shape, 'original output', y.shape)
print(f"Sample tensor size: {X_tensor.shape, y_tensor.shape}")
print(f"Length of train dataloader: {len(train_dataloader)} batches of {batch_size}")
print('sample data point and label', train_data[120] )

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(2, 8), # input matches with X dimension
            nn.ReLU(),
            nn.Linear(8, 3) # Output layer must have same classes
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

In [ ]:
# Define hyperparameters
learning_rate = 0.01

# Define the loss function (Cross Entropy Loss for multi-class classification)
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer (Adam is a common choice)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch*len(X)
            print(f'loss:{loss:>7f} [{current:>5d}/{size:>5d}]')

In [ ]:
def test(dataloader, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm

epochs = 20
for t in tqdm(range(epochs)):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(train_dataloader, model, loss_fn)
print("Done!")

In [ ]:
# Visual plot the resulting classifier
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Convert grid points to PyTorch tensor
grid_points = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()]).float().to(device)

# Get scores from the trained PyTorch model
model.eval() # Set model to evaluation mode
with torch.no_grad():
    scores = model(grid_points)

# Convert scores back to numpy and predict classes
Z = np.argmax(scores.cpu().numpy(), axis=1)
Z = Z.reshape(xx.shape)

fig = plt.figure()
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())